# ScanMate AI Pro — Colab APK/AAB Builder

Run these cells from top to bottom to build a debug APK, release APK, and release AAB in Google Colab.

The project is offline-safe by default. AI features only call Gemini after the user adds their own API key inside the app.


In [ ]:
# 1) Locate the project folder that contains settings.gradle.kts
from pathlib import Path
import os, shutil, subprocess, textwrap

candidates = [Path('/content/scanmate-ai-pro'), Path('/content/ScanMateAIPro'), Path('/content')]
PROJECT_DIR = None
for candidate in candidates:
    if (candidate / 'settings.gradle.kts').exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    matches = list(Path('/content').rglob('settings.gradle.kts'))
    if matches:
        PROJECT_DIR = matches[0].parent

if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find settings.gradle.kts. Upload/extract the project ZIP under /content first.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('Files:', [p.name for p in PROJECT_DIR.iterdir()][:20])


In [ ]:
%%bash
# 2) Install JDK 17 and Android command-line tools
set -e
apt-get update -y
apt-get install -y openjdk-17-jdk unzip wget curl

export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH=$JAVA_HOME/bin:$PATH
java -version


In [ ]:
# 3) Install Android SDK packages needed by the default build
import os, subprocess, pathlib, shutil
from pathlib import Path

SDK_ROOT = Path('/content/android-sdk')
os.environ['ANDROID_HOME'] = str(SDK_ROOT)
os.environ['ANDROID_SDK_ROOT'] = str(SDK_ROOT)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = f"{os.environ['JAVA_HOME']}/bin:{SDK_ROOT}/cmdline-tools/latest/bin:{SDK_ROOT}/platform-tools:" + os.environ['PATH']

SDK_ROOT.mkdir(parents=True, exist_ok=True)
cmdline = SDK_ROOT / 'cmdline-tools' / 'latest'
if not (cmdline / 'bin' / 'sdkmanager').exists():
    urls = [
        'https://dl.google.com/android/repository/commandlinetools-linux-13114758_latest.zip',
        'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip',
        'https://dl.google.com/android/repository/commandlinetools-linux-10406996_latest.zip',
    ]
    zip_path = Path('/content/cmdline-tools.zip')
    downloaded = False
    for url in urls:
        print('Trying', url)
        result = subprocess.run(['bash', '-lc', f'wget -q -O {zip_path} {url}'], text=True)
        if result.returncode == 0 and zip_path.exists() and zip_path.stat().st_size > 1_000_000:
            downloaded = True
            break
    if not downloaded:
        raise RuntimeError('Could not download Android command-line tools. Check Colab internet access.')
    subprocess.check_call(['bash', '-lc', f'rm -rf {SDK_ROOT}/cmdline-tools && mkdir -p {SDK_ROOT}/cmdline-tools && unzip -q -o {zip_path} -d {SDK_ROOT}/cmdline-tools'])
    extracted = SDK_ROOT / 'cmdline-tools' / 'cmdline-tools'
    cmdline.parent.mkdir(parents=True, exist_ok=True)
    if extracted.exists():
        if cmdline.exists():
            shutil.rmtree(cmdline)
        extracted.rename(cmdline)

subprocess.run(['bash', '-lc', 'yes | sdkmanager --licenses >/dev/null'], check=False)
subprocess.check_call(['bash', '-lc', 'sdkmanager "platform-tools" "platforms;android-35" "build-tools;35.0.0"'])
print('Android SDK ready:', SDK_ROOT)


In [ ]:
# 4) Optional: configure release signing. Leave blank for unsigned release outputs.
import os

# Example for Google Drive keystore after mounting Drive:
# os.environ['KEYSTORE_PATH'] = '/content/drive/MyDrive/scanmate-upload-key.jks'
# os.environ['STORE_PASSWORD'] = 'your_store_password'
# os.environ['KEY_ALIAS'] = 'upload'
# os.environ['KEY_PASSWORD'] = 'your_key_password'

print('Release signing configured:', all(os.environ.get(k) for k in ['KEYSTORE_PATH', 'STORE_PASSWORD', 'KEY_PASSWORD']))


In [ ]:
%%bash
# 5) Build debug APK, release APK, and release AAB
set -e
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export ANDROID_HOME=/content/android-sdk
export ANDROID_SDK_ROOT=/content/android-sdk
export PATH=$JAVA_HOME/bin:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools:$PATH

chmod +x ./gradlew
./scripts/doctor.sh
./gradlew clean --no-daemon
./gradlew assembleDebug --no-daemon --stacktrace
./gradlew assembleRelease --no-daemon --stacktrace
./gradlew bundleRelease --no-daemon --stacktrace


In [ ]:
# 6) Collect APK/AAB outputs into /content/scanmate-outputs
from pathlib import Path
import shutil, os

out_dir = Path('/content/scanmate-outputs')
out_dir.mkdir(parents=True, exist_ok=True)
artifacts = list(Path('app/build/outputs').rglob('*.apk')) + list(Path('app/build/outputs').rglob('*.aab'))
for artifact in artifacts:
    dest = out_dir / artifact.name
    shutil.copy2(artifact, dest)
    print(dest, dest.stat().st_size, 'bytes')

if not artifacts:
    raise FileNotFoundError('No APK/AAB files found. Check the build error above.')


In [ ]:
# 7) Download from the left Files panel or use this helper in Colab
from google.colab import files
from pathlib import Path
for artifact in Path('/content/scanmate-outputs').glob('*'):
    print('Ready:', artifact)
    # Uncomment to trigger browser download one by one:
    # files.download(str(artifact))
